# 선행연구 방식 RUL 예측 — GRU·LSTM (초안)

| 항목 | 내용 |
|---|---|
| 데이터 | `data/processed/prefault/` (고장 24시간 전 원본, 4초 간격) |
| 목적 | 선행연구와 같은 조건으로 고장까지 남은 시간(RUL)을 예측하고, 장비 단위 검증에서도 통하는지 확인 |
| 라이브러리 | PyTorch(GRU·LSTM), LightGBM(비교용) |
| 상태 | 초안 (튜닝 최소) |

## 1. 선행연구에서 가져온 설정

| 설정 | 내용 | 근거 |
|---|---|---|
| 사용 구간 | 셔터 위치 1(식각 중) 행만 | TCS 2018, UMD 2025 |
| 문제 | 고장 종류별로 따로, RUL(초)을 숫자로 예측 | Hitachi 2018, UMD 2025 |
| 예측 범위 | RUL ≤ 5,000초(약 1.4시간)인 고장 직전 구간만 학습·평가 | UMD 2025 ("선행 관행") |
| 입력 센서 | TCS가 고른 9개 + 운전 조건(레시피·스텝)별 z값 9개 = 18개 | TCS 2018, UMD 2025(조건별 z값) |
| 입력 길이 | 직전 60시점(2행마다 1개, 약 8분) | Hitachi는 약 5시간을 다운샘플, 여기서는 CPU 시간 때문에 짧게 |
| 모델 | 2층 GRU·LSTM(은닉 64) | TCS·Hitachi(LSTM), 2021 논문(GRU) |
| 지표 | RMSE·MAE(초) | Hitachi, UMD |

**검증은 두 가지로 한다.**
- **사건 무작위 80/20**: 고장 직전 구간(사건)을 무작위로 나눈다. Hitachi 방식이고, 같은 장비가 학습·평가에 섞인다.
- **장비 단위 4묶음**: 처음 보는 장비로 평가한다. 지금까지 우리 모델 검증과 같은 묶음이다.

**선행연구 수치(참고)**: UMD 2025 표(장비 `01_M02`, 압력 저하만, 사건 하나씩 빼는 검증)에서 LSTM 1,469초, TCLSTM 601초, ATCN-LSTM 597초, 랜덤포레스트 회귀 5,476초. 장비·검증 방식이 달라 직접 비교는 어렵다.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import torch
from torch import nn
import lightgbm as lgb
from IPython.display import display

cwd = Path.cwd().resolve()
ROOT = next(p for p in (cwd, cwd.parent) if (p / 'data/processed/prefault').exists())
PROC = ROOT / 'data/processed'

def show(fig, height=480):
    fig.update_layout(template='plotly_white', height=height, font=dict(family='Malgun Gothic, sans-serif'),
                      margin=dict(l=30, r=30, t=75, b=60))
    display({'application/vnd.plotly.v1+json': json.loads(fig.to_json())}, raw=True)

SENS9 = ['IONGAUGEPRESSURE', 'ETCHBEAMVOLTAGE', 'ETCHBEAMCURRENT', 'ETCHSUPPRESSORVOLTAGE', 'ETCHSUPPRESSORCURRENT',
         'FLOWCOOLFLOWRATE', 'FLOWCOOLPRESSURE', 'ETCHGASCHANNEL1READBACK', 'ETCHPBNGASREADBACK']
FAULTS = ['FlowCool Pressure Dropped Below Limit', 'Flowcool Pressure Too High Check Flowcool Pump', 'Flowcool leak']
FSHORT = dict(zip(FAULTS, ['압력 저하', '압력 과다', '누설']))
TTF = {f: 'TTF_' + f for f in FAULTS}
CAP = 5000          # 예측 범위(초)
L, DS = 60, 2       # 입력 길이, 다운샘플 간격
STRIDE = 10         # 목표 시점을 10행마다 하나씩
SEED = 0
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.set_num_threads(8)
print('torch', torch.__version__, '| 장치', DEVICE)

def label(tool):
    return f'{tool[2:]}_{tool[:2]}'

torch 2.15.0+cu132 | 장치 cuda


## 2. 데이터 준비

In [2]:
cols = ['Tool', 'time', 'recipe', 'recipe_step', 'FIXTURESHUTTERPOSITION', *SENS9, *TTF.values()]
pf = pd.concat([pd.read_parquet(f, columns=cols) for f in sorted((PROC / 'prefault').glob('*.parquet'))], ignore_index=True)
n_all = len(pf)
pf = pf[pf.FIXTURESHUTTERPOSITION == 1].drop(columns='FIXTURESHUTTERPOSITION')
pf['Tool'] = pf.Tool.map(label)
pf['_rs'] = pf.recipe_step.fillna(-1)
pf = pf.sort_values(['Tool', 'time']).reset_index(drop=True)
tool_codes, tool_names = pd.factorize(pf.Tool, sort=True)
starts = pd.Series(np.arange(len(pf))).groupby(tool_codes).transform('min').to_numpy()

# 운전 조건(레시피·스텝)별 평소 기준: step/ 파일의 셔터 1, 고장에서 24시간 넘게 떨어진 구간
scols = ['Tool', 'recipe', 'recipe_step', 'shutter', *TTF.values(), *[s + '_mean' for s in SENS9]]
stp = pd.concat([pd.read_parquet(f, columns=scols) for f in sorted((PROC / 'step').glob('*.parquet'))], ignore_index=True)
far = stp[list(TTF.values())].apply(lambda c: c.isna() | (c > 86400)).all(axis=1)
stp = stp[(stp.shutter == 1) & far].copy()
stp['_rs'] = stp.recipe_step.fillna(-1)
agg = {'n': ('recipe', 'size')}
for s in SENS9:
    stp[s + '_sq'] = stp[s + '_mean'].astype('float64') ** 2
    agg[s + '_sum'] = (s + '_mean', 'sum'); agg[s + '_sq'] = (s + '_sq', 'sum')
BASE = stp.groupby(['Tool', 'recipe', '_rs']).agg(**agg)

def zscore_features(train_tools):
    b = BASE[BASE.index.get_level_values('Tool').isin(train_tools)].groupby(['recipe', '_rs']).sum()
    key = pd.MultiIndex.from_frame(pf[['recipe', '_rs']])
    out = np.zeros((len(pf), len(SENS9)), dtype='float32')
    for k, s in enumerate(SENS9):
        mean = b[s + '_sum'] / b.n
        std = np.sqrt((b[s + '_sq'] / b.n - mean ** 2).clip(lower=0))
        std = std.where((b.n >= 30) & (std > 1e-6))
        z = (pf[s].to_numpy() - mean.reindex(key).to_numpy()) / std.reindex(key).to_numpy()
        out[:, k] = np.nan_to_num(np.clip(z, -10, 10), nan=0.0)
    return out

RAW = np.nan_to_num(pf[SENS9].to_numpy(dtype='float32'), nan=0.0)

windows = {}
for f in FAULTS:
    y = pf[TTF[f]].to_numpy()
    idx = np.flatnonzero(y <= CAP)
    idx = idx[idx - (L - 1) * DS >= starts[idx]][::STRIDE]
    ev = pf.Tool.to_numpy()[idx] + '_' + np.round(pf.time.to_numpy()[idx] + y[idx]).astype('int64').astype(str)
    windows[f] = dict(idx=idx, y=y[idx].astype('float32'), ev=ev, tool=pf.Tool.to_numpy()[idx])

print(f'prefault {n_all:,}행 중 셔터 1: {len(pf):,}행')
display(pd.DataFrame({FSHORT[f]: {'윈도우': len(w['idx']), '사건': len(set(w['ev'])), '장비': len(set(w['tool'])),
                                  'RUL 평균(초)': round(float(w['y'].mean()))} for f, w in windows.items()}).T)

prefault 6,384,163행 중 셔터 1: 3,576,034행


,윈도우,사건,장비,RUL 평균(초)
압력 저하,24498,502,19,2554
압력 과다,5007,114,15,2606
누설,4660,74,12,2553


## 3. 모델과 학습 함수

- **평균 예측**: 학습 구간 RUL의 평균을 항상 내놓는다. 모델이 이보다 나아야 의미가 있다.
- **LightGBM**: 윈도우의 마지막 값·평균·(마지막−처음)을 요약 피처로 쓰는 머신러닝 기준선.
- **GRU·LSTM**: 윈도우(60시점 × 18개)를 그대로 넣는다. 목표는 RUL/5000. 학습 사건의 10%로 조기 종료.

In [3]:
OFFS = DS * np.arange(L)[::-1]

def gather(X, idx):
    return X[idx[:, None] - OFFS[None, :]]           # (n, L, F)

class RNN(nn.Module):
    def __init__(self, n_in, kind='gru', h=64):
        super().__init__()
        cell = nn.GRU if kind == 'gru' else nn.LSTM
        self.rnn = cell(n_in, h, num_layers=2, batch_first=True, dropout=0.2)
        self.head = nn.Sequential(nn.Linear(h, 32), nn.ReLU(), nn.Linear(32, 1))
    def forward(self, x):
        o, _ = self.rnn(x)
        return self.head(o[:, -1]).squeeze(-1)

def fit_rnn(Xtr, ytr, Xva, yva, kind, epochs=15, patience=3, bs=512):
    torch.manual_seed(SEED)
    model = RNN(Xtr.shape[2], kind).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    Xtr_t, ytr_t = torch.from_numpy(Xtr), torch.from_numpy(ytr / CAP)
    Xva_t = torch.from_numpy(Xva).to(DEVICE)
    best, best_state, bad = np.inf, None, 0
    g = torch.Generator().manual_seed(SEED)
    for ep in range(epochs):
        model.train()
        perm = torch.randperm(len(Xtr_t), generator=g)
        for i in range(0, len(perm), bs):
            b = perm[i:i + bs]
            opt.zero_grad()
            loss = nn.functional.mse_loss(model(Xtr_t[b].to(DEVICE)), ytr_t[b].to(DEVICE))
            loss.backward(); opt.step()
        model.eval()
        with torch.no_grad():
            va = float(np.sqrt(np.mean((model(Xva_t).cpu().numpy() * CAP - yva) ** 2)))
        if va < best - 1:
            best, best_state, bad = va, {k: v.clone() for k, v in model.state_dict().items()}, 0
        else:
            bad += 1
            if bad >= patience:
                break
    model.load_state_dict(best_state)
    return model, ep + 1

def predict_rnn(model, X, bs=4096):
    model.eval(); out = []
    with torch.no_grad():
        for i in range(0, len(X), bs):
            out.append(model(torch.from_numpy(X[i:i + bs]).to(DEVICE)).cpu().numpy())
    return np.clip(np.concatenate(out) * CAP, 0, CAP)

def summary_feats(W):
    return np.concatenate([W[:, -1], W.mean(axis=1), W[:, -1] - W[:, 0]], axis=1)

def metrics(y, p):
    return round(float(np.sqrt(np.mean((p - y) ** 2)))), round(float(np.mean(np.abs(p - y))))

## 4. 실행

In [4]:
rng = np.random.default_rng(SEED)
all_tools = list(tool_names)
perm_t = rng.permutation(sorted(all_tools, key=lambda t: f'{t[4:]}{t[:3]}'))
tool_folds = [list(perm_t[k::4]) for k in range(4)]

def splits(w):
    evs = np.array(sorted(set(w['ev'])))
    r = np.random.default_rng(SEED).permutation(evs)
    test_ev = set(r[:int(len(r) * 0.2)])
    yield '사건 무작위 80/20', 1, np.array([e in test_ev for e in w['ev']]), all_tools
    for k, tt in enumerate(tool_folds):
        yield '장비 단위 4묶음', k + 1, np.isin(w['tool'], tt), [t for t in all_tools if t not in tt]

rows, preds = [], []
Z_cache = {}
for f in FAULTS:
    w = windows[f]
    kinds = ['gru', 'lstm'] if f == FAULTS[0] else ['gru']
    for split, k, is_test, train_tools in splits(w):
        key = tuple(sorted(train_tools))
        if key not in Z_cache:
            Z_cache[key] = zscore_features(train_tools)
        X = np.concatenate([RAW, Z_cache[key]], axis=1)
        W = gather(X, w['idx'])
        tr, te = ~is_test, is_test
        if te.sum() == 0 or tr.sum() == 0:
            continue
        tr_ev = np.array(sorted(set(w['ev'][tr])))
        va_ev = set(np.random.default_rng(SEED + k).permutation(tr_ev)[:max(1, len(tr_ev) // 10)])
        va = tr & np.array([e in va_ev for e in w['ev']])
        fit = tr & ~va
        base = dict(고장=FSHORT[f], 검증=split, 묶음=k, 학습사건=len(set(w['ev'][fit])), 평가사건=len(set(w['ev'][te])),
                    평가윈도우=int(te.sum()))
        y_te = w['y'][te]
        rows.append({**base, '모델': '평균 예측', **dict(zip(['RMSE', 'MAE'], metrics(y_te, np.full_like(y_te, w['y'][fit].mean()))))})
        gbm = lgb.LGBMRegressor(n_estimators=400, learning_rate=0.05, num_leaves=31, min_child_samples=50,
                                subsample=0.8, subsample_freq=1, colsample_bytree=0.8, random_state=SEED, verbose=-1)
        gbm.fit(summary_feats(W[fit]), w['y'][fit])
        p = np.clip(gbm.predict(summary_feats(W[te])), 0, CAP)
        rows.append({**base, '모델': 'LightGBM', **dict(zip(['RMSE', 'MAE'], metrics(y_te, p)))})
        for kind in kinds:
            model, n_ep = fit_rnn(W[fit], w['y'][fit], W[va], w['y'][va], kind)
            p = predict_rnn(model, W[te])
            rows.append({**base, '모델': kind.upper(), '에폭': n_ep, **dict(zip(['RMSE', 'MAE'], metrics(y_te, p)))})
            if kind == 'gru':
                preds.append(pd.DataFrame({'고장': FSHORT[f], '검증': split, 'ev': w['ev'][te], 'y': y_te, 'p': p}))
        print(f'{FSHORT[f]} · {split} · 묶음 {k} 완료', flush=True)
res = pd.DataFrame(rows)
pred = pd.concat(preds, ignore_index=True)

압력 저하 · 사건 무작위 80/20 · 묶음 1 완료


압력 저하 · 장비 단위 4묶음 · 묶음 1 완료


압력 저하 · 장비 단위 4묶음 · 묶음 2 완료


압력 저하 · 장비 단위 4묶음 · 묶음 3 완료


압력 저하 · 장비 단위 4묶음 · 묶음 4 완료


압력 과다 · 사건 무작위 80/20 · 묶음 1 완료


... (5 outputs left out)

누설 · 장비 단위 4묶음 · 묶음 1 완료


누설 · 장비 단위 4묶음 · 묶음 2 완료


누설 · 장비 단위 4묶음 · 묶음 3 완료


누설 · 장비 단위 4묶음 · 묶음 4 완료


### 4-1. 결과 요약

장비 단위는 4묶음의 RMSE·MAE를 평가 윈도우 수로 가중 평균했다. 단위는 초이고, 예측 범위가 0~5,000초라 RMSE 1,443초 정도가 "아무 정보 없이 범위 가운데를 찍는" 수준이다.

In [5]:
def wavg(g, col):
    return round(float(np.average(g[col], weights=g['평가윈도우'])))
summ = (res.groupby(['고장', '검증', '모델'], sort=False)
        .apply(lambda g: pd.Series({'RMSE(초)': wavg(g, 'RMSE'), 'MAE(초)': wavg(g, 'MAE'),
                                    '평가 사건': int(g['평가사건'].sum())}), include_groups=False)
        .reset_index())
base_rmse = summ[summ.모델 == '평균 예측'].set_index(['고장', '검증'])['RMSE(초)']
summ['평균 예측 대비 RMSE 개선(%)'] = [round((1 - r / base_rmse[(a, b)]) * 100, 1) for a, b, r in
                                 zip(summ.고장, summ.검증, summ['RMSE(초)'])]
display(summ)
display(res[res.검증 == '장비 단위 4묶음'].pivot_table(index=['고장', '모델'], columns='묶음', values='RMSE', sort=False))

,고장,검증,모델,RMSE(초),MAE(초),평가 사건,평균 예측 대비 RMSE 개선(%)
0,압력 저하,사건 무작위 80/20,평균 예측,1390,1195,100,0.0
1,압력 저하,사건 무작위 80/20,LightGBM,1340,1086,100,3.6
2,압력 저하,사건 무작위 80/20,GRU,1360,1153,100,2.2
3,압력 저하,사건 무작위 80/20,LSTM,1360,1163,100,2.2
4,압력 저하,장비 단위 4묶음,평균 예측,1424,1224,502,0.0
5,압력 저하,장비 단위 4묶음,LightGBM,1459,1227,502,-2.5
6,압력 저하,장비 단위 4묶음,GRU,1627,1357,502,-14.3
7,압력 저하,장비 단위 4묶음,LSTM,1440,1224,502,-1.1
8,압력 과다,사건 무작위 80/20,평균 예측,1533,1359,22,0.0
9,압력 과다,사건 무작위 80/20,LightGBM,1336,1091,22,12.9


묶음                   1       2       3       4
고장    모델                                      
압력 저하 평균 예측     1416.0  1411.0  1466.0  1445.0
      LightGBM  1537.0  1433.0  1486.0  1456.0
      GRU       1554.0  1717.0  1493.0  1424.0
      LSTM      1514.0  1415.0  1473.0  1430.0
압력 과다 평균 예측     1478.0  1532.0  1559.0  1485.0
      LightGBM  1732.0  1853.0  1795.0  1665.0
      GRU       1585.0  1710.0  1708.0  1584.0
누설    평균 예측     1430.0  1507.0  1455.0  1426.0
      LightGBM  1545.0  1447.0  1624.0  1395.0
      GRU       1550.0  1558.0  1967.0  1432.0

In [6]:
fig = go.Figure()
order = ['평균 예측', 'LightGBM', 'GRU', 'LSTM']
colors = {'평균 예측': '#9aa3ab', 'LightGBM': '#4c9a52', 'GRU': '#2a78d6', 'LSTM': '#eb6834'}
for m in order:
    s = summ[summ.모델 == m]
    if len(s):
        fig.add_bar(x=[f'{a}<br>{b}' for a, b in zip(s.고장, s.검증)], y=s['RMSE(초)'], name=m, marker_color=colors[m])
fig.add_hline(y=597, line=dict(color='#444', dash='dot'), annotation_text='UMD 표 ATCN-LSTM 597초 (장비 1대·압력 저하)')
fig.update_layout(title='그림 4-1. RMSE 비교 (낮을수록 좋음)', barmode='group', yaxis_title='RMSE(초)')
show(fig, 500)

### 4-2. 예측 예시 (GRU, 압력 저하)

In [7]:
fig = go.Figure()
pal = ['#2a78d6', '#eb6834', '#4c9a52']
for split, dash in [('사건 무작위 80/20', 'solid'), ('장비 단위 4묶음', 'dot')]:
    d = pred[(pred.고장 == '압력 저하') & (pred.검증 == split)]
    top = d.ev.value_counts().index[:3]
    for c, e in zip(pal, top):
        g = d[d.ev == e].sort_values('y', ascending=False)
        fig.add_scatter(x=-g.y / 60, y=g.p / 60, mode='lines', line=dict(color=c, dash=dash), name=f'{split} · {e}')
fig.add_scatter(x=[-CAP / 60, 0], y=[CAP / 60, 0], mode='lines', line=dict(color='#999', width=1), name='정답(대각선)')
fig.update_layout(title='그림 4-2. 실제 남은 시간 대비 예측 (사건별, 윈도우가 많은 3개씩)',
                  xaxis_title='실제: 고장까지 남은 시간(분, 음수)', yaxis_title='예측 남은 시간(분)', legend=dict(orientation='h', y=-0.3))
show(fig, 560)

## 5. 해석과 한계

- **선행연구 설정을 따라 해도 센서만으로는 남은 시간을 거의 못 맞춘다.** 압력 저하 기준 사건 무작위 80/20에서 GRU·LSTM RMSE 1,360초, LightGBM 1,340초다. 항상 평균을 내놓는 예측(1,390초)보다 2~4% 나을 뿐이다.
- **장비 단위 검증에서는 평균 예측보다도 나쁘다.** 압력 저하 GRU 1,627초(평균 예측 1,424초), LSTM 1,440초. 압력 과다·누설도 모든 모델이 평균 예측보다 나쁘다. 압력 과다는 사건 무작위에서 LightGBM이 12.9% 나았지만, 평가 사건이 22건뿐이라 우연일 수 있다.
- **선행연구 수치(약 600~1,500초)와 차이가 나는 이유로 추정되는 것**
  - UMD 표는 장비 1대(`01_M02`)·압력 저하만, 사건 하나씩 빼는 검증이다. 같은 장비의 다른 사건으로 학습하므로 우리보다 훨씬 쉬운 조건이다.
  - UMD는 각 사건 구간 시작부터 흐른 시간과 "시간 사전값"을 함께 썼다. RUL ≤ 5,000초로 자른 구간에서는 흐른 시간만 알아도 RUL을 거의 계산할 수 있어서, 센서 신호가 약해도 오차가 작게 나올 수 있다. 우리는 센서 값만 넣었다.
  - Hitachi는 고장 시점에서 거꾸로 일정 간격으로 샘플을 뽑아 시퀀스를 만들었고, 시퀀스를 무작위로 나눴다.
- EDA(`eda/eda_step.ipynb` 4-7)와도 맞는 결과다. 셔터 1 기록에서 고장 직전 몇 분~몇 시간 사이의 냉각 압력 변화는 거의 없었다. 고장의 30~60%는 직전 1시간 기록도 없다.
- 한계: 입력 길이가 약 8분으로 짧다(Hitachi는 약 5시간). 튜닝을 거의 하지 않았다. 셔터 1 행만 이어 붙였으므로 윈도우 사이에 기록이 끊긴 시간이 섞일 수 있다. 평가 사건이 압력 과다 114건, 누설 74건으로 적다.
- 다음 후보: (1) 입력 길이를 1~5시간으로 늘려 비교, (2) 흐른 시간 피처를 넣었을 때 선행연구 수치가 재현되는지 확인(누수 여부 점검), (3) 장비 `01_M02`(`M02_01`)·압력 저하만으로 UMD 조건 재현, (4) 회귀 대신 "N분 안에 고장" 분류와 사건 단위 탐지율·오경보로 평가.